# FlashAttention-3 (FA3) Python Tiling & FP8 Simulation

This notebook demonstrates the next-generation optimizations introduced in **FlashAttention-3** for NVIDIA Hopper GPUs (`sm_90` / H100 / H200):
1. **FP8 Low-Precision Scaling**: Simulating E4M3 quantization, dynamic scaling factor calculations, and output alignment to prevent accuracy loss under low precision.
2. **Asynchronous Pipelining (TMA / WGMMA)**: A conceptual producer-consumer model demonstrating how Hopper's Tensor Memory Accelerator overlaps HBM transfers with Tensor Core calculations.

Let's run the simulations in PyTorch and verify their behavior.

In [1]:
import math
import time
import queue
import threading
import torch

print(f"PyTorch Version: {torch.__version__}")

PyTorch Version: 2.10.0+cpu


## 1. FP8 Quantization and Scaling Simulation

FP8 formats (such as E4M3) have a very limited dynamic range compared to FP16/FP32. To prevent underflow or overflow, we compute dynamic scaling factors $S_x = \frac{\text{max\_fp8}}{\max(|x|)}$ to scale tensors into the optimal range of FP8 before casting.

In [2]:
def quantize_to_fp8_sim(x, max_fp8=448.0):
    """
    Simulates FP8 (E4M3) quantization by scaling the input tensor into the FP8 range,
    clamping to max_fp8, rounding, and returning the scaled tensor and its inverse scale factor.
    """
    # Compute scale factor
    max_val = torch.max(torch.abs(x))
    if max_val == 0:
        scale = 1.0
    else:
        scale = max_fp8 / max_val
        
    # Scale and clamp (simulate casting)
    x_scaled = x * scale
    x_fp8 = torch.clamp(torch.round(x_scaled), -max_fp8, max_fp8)
    
    # Inverse scale to dequantize back to float
    inv_scale = 1.0 / scale
    return x_fp8, inv_scale

# Verification check
test_tensor = torch.randn(5, 5) * 15.0
fp8_sim, inv_s = quantize_to_fp8_sim(test_tensor)
dequantized = fp8_sim * inv_s

print("Original Tensor:")
print(test_tensor[:2, :2])
print("\nSimulated FP8 Dequantized (Max FP8 range scale):")
print(dequantized[:2, :2])
diff = torch.norm(test_tensor - dequantized).item()
print(f"\nQuantization Error (Frobenius Norm): {diff:.4e}")

Original Tensor:
tensor([[ 13.6549, -23.7894],
        [ -1.0396,   9.6203]])

Simulated FP8 Dequantized (Max FP8 range scale):
tensor([[ 13.6050, -23.7835],
        [ -1.0078,   9.5739]])

Quantization Error (Frobenius Norm): 1.3635e-01


## 2. FP8 FlashAttention Tiling Simulation with Scaling

Here we simulate the block-by-block attention computations where $Q_i, K_j, V_j$ are quantized and multiplied in FP8, while the online softmax calculations are done in high precision (FP32) to maintain accuracy.

In [3]:
def flash_attention_v3_fp8_sim(Q, K, V, B_r=64, B_c=64):
    """
    Python simulation of FlashAttention-3 FP8 forward pass with dynamic scaling.
    """
    B, H, N, d = Q.shape
    d_k = d
    O = torch.zeros_like(Q)
    
    for b in range(B):
        for h in range(H):
            Q_bh = Q[b, h]
            K_bh = K[b, h]
            V_bh = V[b, h]
            
            O_bh = torch.zeros((N, d), device=Q.device, dtype=Q.dtype)
            
            # Compute global quantization scale factors for FP8 matrix multiplies
            # (In production, this is tracked using running history metrics)
            Q_fp8, inv_s_q = quantize_to_fp8_sim(Q_bh)
            K_fp8, inv_s_k = quantize_to_fp8_sim(K_bh)
            V_fp8, inv_s_v = quantize_to_fp8_sim(V_bh)
            
            Tr = math.ceil(N / B_r)
            Tc = math.ceil(N / B_c)
            
            for i in range(Tr):
                start_r = i * B_r
                end_r = min(start_r + B_r, N)
                
                Q_i_fp8 = Q_fp8[start_r:end_r, :]  # Block Q (FP8)
                
                S_i = torch.zeros((end_r - start_r, d), device=Q.device)
                m_i = torch.full((end_r - start_r, 1), float('-inf'), device=Q.device)
                l_i = torch.zeros((end_r - start_r, 1), device=Q.device)
                
                for j in range(Tc):
                    start_c = j * B_c
                    end_c = min(start_c + B_c, N)
                    
                    K_j_fp8 = K_fp8[start_c:end_c, :]  # Block K (FP8)
                    V_j_fp8 = V_fp8[start_c:end_c, :]  # Block V (FP8)
                    
                    # 1. Compute scores in FP8, then dequantize back to high precision for softmax
                    scores_ij_fp8 = torch.matmul(Q_i_fp8, K_j_fp8.transpose(-2, -1))
                    scores_ij = (scores_ij_fp8 * inv_s_q * inv_s_k) / math.sqrt(d_k)
                    
                    # 2. High precision Softmax block statistics
                    tilde_m_ij, _ = torch.max(scores_ij, dim=-1, keepdim=True)
                    tilde_P_ij = torch.exp(scores_ij - tilde_m_ij)
                    tilde_l_ij = torch.sum(tilde_P_ij, dim=-1, keepdim=True)
                    
                    # 3. Scale statistics updates
                    m_i_new = torch.max(m_i, tilde_m_ij)
                    alpha = torch.exp(m_i - m_i_new)
                    beta = torch.exp(tilde_m_ij - m_i_new)
                    
                    l_i = alpha * l_i + beta * tilde_l_ij
                    
                    # 4. Quantize attention probabilities back to FP8 to perform AV GEMM
                    # Simulate FP8 scaling on attention weights slice
                    P_ij_scaled = beta * tilde_P_ij
                    P_ij_fp8, inv_s_p = quantize_to_fp8_sim(P_ij_scaled)
                    
                    # Compute output block using FP8 matrix multiply
                    output_ij_fp8 = torch.matmul(P_ij_fp8, V_j_fp8)
                    output_ij = output_ij_fp8 * inv_s_p * inv_s_v
                    
                    # Accumulate outputs
                    S_i = alpha * S_i + output_ij
                    m_i = m_i_new
                    
                O_bh[start_r:end_r, :] = S_i / l_i
            
            O[b, h] = O_bh
            
    return O

### Verification of FP8 FlashAttention Correctness

In [4]:
Q = torch.randn(2, 4, 256, 64)
K = torch.randn(2, 4, 256, 64)
V = torch.randn(2, 4, 256, 64)

out_std = Q.new_zeros(Q.shape)  # Baseline standard attention
d_k = Q.shape[-1]
for b in range(Q.shape[0]):
    for h in range(Q.shape[1]):
        scores = torch.matmul(Q[b, h], K[b, h].transpose(-2, -1)) / math.sqrt(d_k)
        out_std[b, h] = torch.matmul(torch.softmax(scores, dim=-1), V[b, h])
        
out_fp8 = flash_attention_v3_fp8_sim(Q, K, V, B_r=32, B_c=32)

match = torch.allclose(out_std, out_fp8, atol=1e-2, rtol=1e-2)  # Higher tolerance for FP8 approximation
print(f"Standard Float32 Attention vs. FP8 FlashAttention-3 Match (tol=1e-2): {match}")
diff = torch.norm(out_std - out_fp8).item()
print(f"Difference Norm: {diff:.4e}")

Standard Float32 Attention vs. FP8 FlashAttention-3 Match (tol=1e-2): True
Difference Norm: 2.2422e-01


## 3. Producer-Consumer Asynchronous Pipelining Simulation

In Hopper GPUs, the **Tensor Memory Accelerator (TMA)** acts as a producer that loads memory asynchronously without consuming core execution cycles. The **WGMMA Tensor Cores** act as consumers that execute matrix multiplies.

The following code simulates this pipelining mechanism using threads and queues to demonstrate how compute latency gets completely hidden behind memory access overlapping.

In [5]:
shared_sram = queue.Queue(maxsize=2)  # Double-buffering SRAM (Ping-Pong Buffer)
num_blocks = 5

def producer_tma():
    """
    Simulates Hopper's TMA loading blocks of K/V asynchronously from HBM to SRAM.
    """
    for block_idx in range(num_blocks):
        print(f"[TMA (Producer)] Starting load of Block {block_idx} from HBM to SRAM...")
        time.sleep(0.15)  # Simulate HBM transfer latency
        shared_sram.put(f"Block_{block_idx}_Data")
        print(f"[TMA (Producer)] Block {block_idx} loaded into SRAM buffer.\n")
        
def consumer_wgmma():
    """
    Simulates Hopper's WGMMA executing matrix multiplies directly out of SRAM.
    """
    for block_idx in range(num_blocks):
        # Retrieve data from SRAM buffer
        data = shared_sram.get()
        print(f"  [WGMMA (Consumer)] Grabbed {data} from SRAM. Computing GEMMs...")
        time.sleep(0.08)  # GEMM computations are fast on Tensor Cores
        print(f"  [WGMMA (Consumer)] Completed GEMM compute for {data}.")
        shared_sram.task_done()

# Start pipeline
start_time = time.time()
tma_thread = threading.Thread(target=producer_tma)
wgmma_thread = threading.Thread(target=consumer_wgmma)

tma_thread.start()
wgmma_thread.start()

tma_thread.join()
wgmma_thread.join()

total_pipeline_time = time.time() - start_time
serial_time = (0.15 + 0.08) * num_blocks
print("=" * 70)
print(f"Pipeline Time: {total_pipeline_time:.4f}s (vs. Serial Time: {serial_time:.4f}s)")
print(f"Hidden Latency: {serial_time - total_pipeline_time:.4f}s saved by overlapping!")
print("=" * 70)

[TMA (Producer)] Starting load of Block 0 from HBM to SRAM...
[TMA (Producer)] Block 0 loaded into SRAM buffer.

[TMA (Producer)] Starting load of Block 1 from HBM to SRAM...
  [WGMMA (Consumer)] Grabbed Block_0_Data from SRAM. Computing GEMMs...
  [WGMMA (Consumer)] Completed GEMM compute for Block_0_Data.
[TMA (Producer)] Block 1 loaded into SRAM buffer.

[TMA (Producer)] Starting load of Block 2 from HBM to SRAM...
  [WGMMA (Consumer)] Grabbed Block_1_Data from SRAM. Computing GEMMs...
  [WGMMA (Consumer)] Completed GEMM compute for Block_1_Data.
[TMA (Producer)] Block 2 loaded into SRAM buffer.

[TMA (Producer)] Starting load of Block 3 from HBM to SRAM...
  [WGMMA (Consumer)] Grabbed Block_2_Data from SRAM. Computing GEMMs...
  [WGMMA (Consumer)] Completed GEMM compute for Block_2_Data.
[TMA (Producer)] Block 3 loaded into SRAM buffer.

[TMA (Producer)] Starting load of Block 4 from HBM to SRAM...
  [WGMMA (Consumer)] Grabbed Block_3_Data from SRAM. Computing GEMMs...
  [WGMMA (Co